## Feature engineering

Antes que nada, identificamos dos cosas muy importantes en el EDA.
1) El pico de 2022 debido a la guerra en Ucrania, esto da precios excepcionales que nada tienen que ver dentro de previsiones 'normales'. Como solucion, los precios se caparan para que no llegue a techos de mas de 600 cuando no es lo normal salvo factores externos.
2) Derivado de la guerra, se puso en marcha el mecanismo de Excepción Ibérica 2022 por el que se desacoplan los precios del gas del mercado eléctrico. Como consecuencia, se observan dos modelos o patrones claramente diferenciados antes de 2022 y despúes de 2022. Antes, los precios son mas estables y menos volátiles debido a que todo se pagaba a precio de gas. Tras el 2022 se paga por producción. A partir de entonces el precio es muy volátil, marcando gráficas de 'forma de pato' típicas de mercados renovables. Además, el precio de las renovables da como resultado ciertos días de precios negativos. Como conclusión, se introducirá una variable binaria (0 pre 2022 y 1 post 2022) para que los modelos distingan los dos regímenes.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(base_dir, "data", "electricity.db")
conn = sqlite3.connect(db_path)

df = pd.read_sql("""
    SELECT * FROM omie_prices_historic
    WHERE hour BETWEEN 1 AND 24
    ORDER BY datetime
""", conn)

print(df.shape)

(52608, 6)


In [2]:
# Capping al percentil 99.5
cap = df["price_es"].quantile(0.995)
df["price_es"] = df["price_es"].clip(upper=cap)
print(f"El capping se hace a un precio de {cap}")

El capping se hace a un precio de 331.92990000000003


In [3]:
# Variable de régimen post-excepcion iberica
# Se hace datetime porque sqlite devuelve str, no datetime format
df["datetime"] = pd.to_datetime(df["datetime"])
# Entra en junio 2022
excepcion_iberica = pd.Timestamp("2022-06-15")
df["post_excepcion"] = (df["datetime"] >= excepcion_iberica).astype(int)

print(df["post_excepcion"].value_counts())

post_excepcion
0    30264
1    22344
Name: count, dtype: int64


### Variables cíclicas

In [4]:
# Variables cíclicas
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["day_of_year"] = df["datetime"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

df["weekday"] = df["datetime"].dt.dayofweek
df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# Lag features
df = df.sort_values("datetime").reset_index(drop=True)
df["lag_1"] = df["price_es"].shift(1)
df["lag_24"] = df["price_es"].shift(24)
df["lag_168"] = df["price_es"].shift(168)

# Medias móviles
df["rolling_mean_24"] = df["price_es"].shift(1).rolling(window=24).mean()
df["rolling_mean_168"] = df["price_es"].shift(1).rolling(window=168).mean()

# Eliminar NaN
df = df.dropna().reset_index(drop=True)

print(f"Filas finales: {len(df)}")
print(f"Columnas: {len(df.columns)}")

Filas finales: 51426
Columnas: 21


In [5]:
df.sample()

,hour,price_es,year,month,day,datetime,post_excepcion,hour_sin,hour_cos,day_of_year,...,doy_cos,weekday,weekday_sin,weekday_cos,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_mean_168
12646,1,34.53,2020,7,2,2020-07-02,0,0.258819,0.965926,184,...,-0.999667,3,0.433884,-0.900969,0,37.91,40.44,40.75,37.454167,35.659702


In [6]:
df.columns

Index(['hour', 'price_es', 'year', 'month', 'day', 'datetime',
       'post_excepcion', 'hour_sin', 'hour_cos', 'day_of_year', 'doy_sin',
       'doy_cos', 'weekday', 'weekday_sin', 'weekday_cos', 'is_weekend',
       'lag_1', 'lag_24', 'lag_168', 'rolling_mean_24', 'rolling_mean_168'],
      dtype='object')

In [7]:
df.to_sql("omie_features_historic", conn, if_exists="replace", index=False)
# Verificar tablas
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,omie_prices
1,omie_features
2,omie_prices_historic
3,weather_Mad_Sev_Zar
4,omie_features_weather
5,omie_features_historic


In [8]:
conn.close()

## 2. Weather Features
Adding meteorological data from Open-Meteo for Madrid, Sevilla and Zaragoza.

In [9]:
conn = sqlite3.connect(db_path)

df_weather = pd.read_sql("""
    SELECT * FROM weather_Mad_Sev_Zar
    ORDER BY datetime
""", conn)

conn.close()

df_weather["datetime"] = pd.to_datetime(df_weather["datetime"])

print(df_weather.shape)
print(df_weather.head())

(52608, 10)
             datetime  solar_radiation_madrid  solar_radiation_sevilla  \
0 2019-01-01 00:00:00                     0.0                      0.0   
1 2019-01-01 01:00:00                     0.0                      0.0   
2 2019-01-01 02:00:00                     0.0                      0.0   
3 2019-01-01 03:00:00                     0.0                      0.0   
4 2019-01-01 04:00:00                     0.0                      0.0   

   solar_radiation_zaragoza  temperature_madrid  temperature_sevilla  \
0                       0.0                 3.7                  9.1   
1                       0.0                 2.0                  8.3   
2                       0.0                 1.1                  7.8   
3                       0.0                -0.1                  7.5   
4                       0.0                -0.6                  7.1   

   temperature_zaragoza  wind_speed_madrid  wind_speed_sevilla  \
0                   5.5                6.2  

Ahora a hacer merge por datetime. Si luego quedan filas sin precio (faltan datos en algunos dias pero los hay de meteo) ya se eliminaran

In [10]:
# Merge — inner join para quedarnos solo con horas que tienen precio Y meteo
df_full = df.merge(df_weather, on="datetime", how="inner")

print(f"Features sin meteo: {len(df)}")
print(f"Features con meteo: {len(df_full)}")
print(f"Columnas totales: {len(df_full.columns)}")

Features sin meteo: 51426
Features con meteo: 51426
Columnas totales: 30


In [11]:
conn = sqlite3.connect(db_path)
df_full.to_sql("omie_features_weather", conn, if_exists="replace", index=False)
print(f"Tabla omie_features_weather creada con {df_full.shape[0]} filas y {df_full.shape[1]} columnas")

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

Tabla omie_features_weather creada con 51426 filas y 30 columnas


,name
0,omie_prices
1,omie_features
2,omie_prices_historic
3,weather_Mad_Sev_Zar
4,omie_features_historic
5,omie_features_weather


In [12]:
conn.close()